In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="03-google-street-view",
    notebook_name="02_object_detection_deep_dive.ipynb"
)

# Object Detection Deep Dive -- From "Where Is Waldo?" to Faster R-CNN

## What This Notebook Covers

In the previous notebook, we designed the full Google Street View blurring system at a high level. Now we zoom in on the **heart** of the system: the object detection model. Think of it this way -- the previous notebook was the architect's blueprint for the whole building. This notebook is the detailed engineering plan for the elevator that makes it all work.

We will cover:
1. Detection vs. Classification vs. Segmentation -- what is the difference?
2. Bounding boxes and how they work
3. Anchor boxes -- the clever trick that makes detection fast
4. The R-CNN family tree: R-CNN -> Fast R-CNN -> Faster R-CNN (evolution with diagrams)
5. Region Proposal Networks (RPN) -- the key innovation
6. One-stage detectors: YOLO and SSD
7. When to use which architecture
8. The combined loss function
9. Data augmentation with bounding box transforms

---

## Staff-Level Technical Summary

Object detection combines localization (regression) and classification into a single pipeline. Two-stage detectors (Faster R-CNN) decouple region proposal from classification, achieving higher accuracy at the cost of speed. One-stage detectors (YOLO, SSD) perform both tasks in a single forward pass, trading some accuracy for real-time performance. The choice depends on whether your system is latency-bound (choose one-stage) or accuracy-bound (choose two-stage). For Google Street View's offline privacy pipeline, we choose Faster R-CNN because recall is paramount and latency is irrelevant.

## 1. Detection vs. Classification vs. Segmentation

### The 12-Year-Old Version

Imagine you have a photo of a park with dogs, cats, and people:

- **Classification**: "There is a dog in this photo." (Just names the thing -- no location.)
- **Object Detection**: "There is a dog at position (100, 50) with size 80x60." (Names it AND draws a box around it.)
- **Segmentation**: "These exact pixels belong to the dog." (Colors in every pixel of the dog, like a perfect sticker cutout.)

Each task builds on the previous one -- more information, more compute, more complexity.

### Why Object Detection for Street View?

We need detection (not just classification) because we need to know **where** each face is so we can blur that specific region. Classification would only tell us "there is a face somewhere" -- useless for blurring. Segmentation would give us pixel-perfect masks, but that is overkill -- a bounding box is enough to blur the area.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch

# === Visualize the three computer vision tasks side by side ===

np.random.seed(42)

# Create a synthetic "scene" image
def create_scene():
    img = np.zeros((250, 400, 3), dtype=np.uint8)
    # Sky gradient
    for i in range(120):
        img[i, :] = [135 + i//3, 206 - i//4, 235 - i//5]
    # Ground
    img[120:, :] = [80, 140, 80]
    # "Person" (rectangular shape)
    img[50:150, 60:110] = [220, 180, 150]  # body
    img[30:55, 70:100] = [210, 170, 140]   # head
    # "Car" shape
    img[100:160, 220:340] = [60, 60, 180]   # car body
    img[80:105, 240:320] = [80, 80, 200]    # car top
    img[145:165, 230:260] = [40, 40, 40]    # wheel
    img[145:165, 300:330] = [40, 40, 40]    # wheel
    # License plate on car
    img[130:145, 260:310] = [240, 240, 240]
    return img

img = create_scene()
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Classification ---
axes[0].imshow(img)
axes[0].set_title('Classification\n"What is in the image?"', fontsize=13, fontweight='bold')
axes[0].text(200, 230, 'Labels: person, car', fontsize=11, ha='center',
             bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.9))
axes[0].axis('off')

# --- Object Detection ---
axes[1].imshow(img)
axes[1].set_title('Object Detection\n"What + Where (bounding box)?"', fontsize=13, fontweight='bold')
# Person bbox
rect1 = patches.Rectangle((55, 25), 60, 130, linewidth=2.5, edgecolor='red', facecolor='none')
axes[1].add_patch(rect1)
axes[1].text(55, 20, 'person 0.95', color='red', fontsize=10, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
# Car bbox
rect2 = patches.Rectangle((215, 75), 130, 95, linewidth=2.5, edgecolor='blue', facecolor='none')
axes[1].add_patch(rect2)
axes[1].text(215, 70, 'car 0.92', color='blue', fontsize=10, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
# Plate bbox
rect3 = patches.Rectangle((255, 127), 60, 22, linewidth=2, edgecolor='green', facecolor='none')
axes[1].add_patch(rect3)
axes[1].text(255, 122, 'plate 0.88', color='green', fontsize=9, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
axes[1].axis('off')

# --- Segmentation ---
axes[2].imshow(img)
axes[2].set_title('Segmentation\n"Exact pixel mask?"', fontsize=13, fontweight='bold')
# Create colored overlay masks
mask_person = np.zeros((*img.shape[:2], 4))
mask_person[30:55, 70:100] = [1, 0, 0, 0.4]  # head
mask_person[50:150, 60:110] = [1, 0, 0, 0.4]  # body
axes[2].imshow(mask_person)
mask_car = np.zeros((*img.shape[:2], 4))
mask_car[80:105, 240:320] = [0, 0, 1, 0.4]
mask_car[100:160, 220:340] = [0, 0, 1, 0.4]
axes[2].imshow(mask_car)
axes[2].axis('off')

plt.suptitle('Three Computer Vision Tasks', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Task Comparison:")
print("  Classification : INPUT=image, OUTPUT=class label(s)")
print("  Detection      : INPUT=image, OUTPUT=list of (class, bounding box, confidence)")
print("  Segmentation   : INPUT=image, OUTPUT=pixel-level class mask")
print()
print("For Street View blurring, DETECTION is the sweet spot:")
print("  - More info than classification (we need location to blur)")
print("  - Less compute than segmentation (a box is good enough for blurring)")

## 2. Bounding Box Representation

### The 12-Year-Old Version

A bounding box is just a rectangle that tells you "the object is inside this rectangle." It is the simplest way to say WHERE something is in a picture.

There are two common ways to describe a rectangle:

1. **(x, y, w, h)** -- top-left corner + width + height (used by COCO dataset)
2. **(x_min, y_min, x_max, y_max)** -- top-left corner + bottom-right corner (used by PASCAL VOC)

Converting between them is easy, and you WILL need to do it in production code.

In [ ]:
# === Bounding Box Formats and Conversion ===

def xywh_to_xyxy(box):
    """Convert (x, y, w, h) to (x_min, y_min, x_max, y_max)."""
    x, y, w, h = box
    return [x, y, x + w, y + h]

def xyxy_to_xywh(box):
    """Convert (x_min, y_min, x_max, y_max) to (x, y, w, h)."""
    x_min, y_min, x_max, y_max = box
    return [x_min, y_min, x_max - x_min, y_max - y_min]

def xywh_to_cxcywh(box):
    """Convert (x, y, w, h) to center format (cx, cy, w, h)."""
    x, y, w, h = box
    return [x + w/2, y + h/2, w, h]

def cxcywh_to_xywh(box):
    """Convert center format (cx, cy, w, h) to (x, y, w, h)."""
    cx, cy, w, h = box
    return [cx - w/2, cy - h/2, w, h]


# Demonstrate with a face bounding box
face_xywh = [100, 60, 80, 100]  # COCO format: top-left (100,60), width=80, height=100

face_xyxy = xywh_to_xyxy(face_xywh)
face_center = xywh_to_cxcywh(face_xywh)

print("=== Bounding Box Format Conversion ===")
print(f"  (x, y, w, h)     = {face_xywh}   <-- COCO format")
print(f"  (xmin,ymin,xmax,ymax) = {face_xyxy}  <-- PASCAL VOC format")
print(f"  (cx, cy, w, h)   = {face_center}    <-- Center format (used by YOLO)")
print()

# Round-trip test
assert xyxy_to_xywh(xywh_to_xyxy(face_xywh)) == face_xywh, "Round trip failed!"
assert cxcywh_to_xywh(xywh_to_cxcywh(face_xywh)) == face_xywh, "Round trip failed!"
print("Round-trip conversion test: PASSED")

# Visualize all three formats
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
img = create_scene()

formats = [
    ("COCO: (x, y, w, h)", face_xywh, f"x={face_xywh[0]}, y={face_xywh[1]}\nw={face_xywh[2]}, h={face_xywh[3]}"),
    ("VOC: (xmin, ymin, xmax, ymax)", face_xywh, f"xmin={face_xyxy[0]}, ymin={face_xyxy[1]}\nxmax={face_xyxy[2]}, ymax={face_xyxy[3]}"),
    ("Center: (cx, cy, w, h)", face_xywh, f"cx={face_center[0]}, cy={face_center[1]}\nw={face_center[2]}, h={face_center[3]}"),
]

for ax, (title, box, label_text) in zip(axes, formats):
    ax.imshow(img)
    x, y, w, h = box
    rect = patches.Rectangle((x, y), w, h, linewidth=2.5, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    
    # Mark the reference point
    if 'Center' in title:
        cx, cy = x + w/2, y + h/2
        ax.plot(cx, cy, 'ro', markersize=8)
        ax.annotate('center', (cx, cy), textcoords='offset points', xytext=(10, -15),
                    fontsize=9, color='red', fontweight='bold')
    else:
        ax.plot(x, y, 'ro', markersize=8)
        ax.annotate('(x, y)', (x, y), textcoords='offset points', xytext=(5, -15),
                    fontsize=9, color='red', fontweight='bold')
    
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.text(200, 230, label_text, fontsize=10, ha='center',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    ax.axis('off')

plt.suptitle('Three Common Bounding Box Formats', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Anchor Boxes -- The Clever Trick

### The 12-Year-Old Version

Imagine you are playing a guessing game. Instead of guessing where a face is from scratch ("Hmm, maybe at pixel 142, 87, with width 53..."), you start with a bunch of **pre-made guess templates** placed all over the image. Some are tall and skinny (good for people standing up), some are wide and short (good for license plates), some are square (good for faces).

The model's job is much easier now: instead of inventing a box from nothing, it just says:
- "This template at position (5, 3) -- YES, there is a face. Nudge it 2 pixels left and make it 5% wider."
- "This template at position (8, 7) -- NOPE, nothing here."

These pre-made templates are called **anchor boxes** (or "priors" or "default boxes").

### Staff-Level Detail

Anchors are generated at each spatial location of the feature map. At each location, we place `k` anchors with different aspect ratios and scales. For Faster R-CNN, the original paper uses 3 scales x 3 ratios = 9 anchors per location. If the feature map is 50x50, that is 50 x 50 x 9 = **22,500 anchors** per image. The model predicts:
1. **Objectness score**: probability that this anchor contains ANY object (binary)
2. **Box deltas**: (dx, dy, dw, dh) offsets to refine the anchor into a tighter box

In [ ]:
# === Visualize Anchor Boxes ===

def generate_anchors(center_x, center_y, scales=[32, 64, 128], ratios=[0.5, 1.0, 2.0]):
    """
    Generate anchor boxes at a single location.
    
    12-year-old version:
    At one spot on the image, we place 9 different-shaped rectangles:
    3 sizes (small, medium, large) x 3 shapes (wide, square, tall) = 9 anchors.
    
    Staff-level detail:
    scale = sqrt(area), ratio = height/width
    For each (scale, ratio) pair:
        w = scale * sqrt(ratio)
        h = scale / sqrt(ratio)
    """
    anchors = []
    for scale in scales:
        for ratio in ratios:
            w = scale * np.sqrt(ratio)
            h = scale / np.sqrt(ratio)
            anchors.append({
                'x': center_x - w/2,
                'y': center_y - h/2,
                'w': w,
                'h': h,
                'scale': scale,
                'ratio': ratio
            })
    return anchors


# Generate anchors at center of a 400x300 image
center_x, center_y = 200, 150
anchors = generate_anchors(center_x, center_y)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71']
scale_names = ['Small (32px)', 'Medium (64px)', 'Large (128px)']
ratio_styles = ['-', '--', ':']
ratio_names = ['wide (0.5)', 'square (1.0)', 'tall (2.0)']

for s_idx, scale in enumerate([32, 64, 128]):
    ax = axes[s_idx]
    ax.set_xlim(0, 400)
    ax.set_ylim(0, 300)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.set_title(f'{scale_names[s_idx]}', fontsize=12, fontweight='bold')
    
    # Draw center point
    ax.plot(center_x, center_y, 'k+', markersize=15, markeredgewidth=2)
    
    for r_idx, ratio in enumerate([0.5, 1.0, 2.0]):
        w = scale * np.sqrt(ratio)
        h = scale / np.sqrt(ratio)
        rect = patches.Rectangle(
            (center_x - w/2, center_y - h/2), w, h,
            linewidth=2.5, edgecolor=colors[r_idx], facecolor='none',
            linestyle=ratio_styles[r_idx]
        )
        ax.add_patch(rect)
    
    ax.set_facecolor('#f5f5f5')
    ax.grid(True, alpha=0.2)

# Add legend
legend_elements = [
    patches.Patch(edgecolor=colors[i], facecolor='none', linestyle=ratio_styles[i],
                  linewidth=2, label=ratio_names[i])
    for i in range(3)
]
axes[2].legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.suptitle('Anchor Boxes at a Single Location\n3 scales x 3 aspect ratios = 9 anchors per location',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Anchor Box Statistics:")
print(f"  Anchors per location: {len(anchors)}")
print(f"  Feature map 50x50 -> Total anchors: {50*50*len(anchors):,}")
print()
for a in anchors:
    print(f"  scale={a['scale']:>3}, ratio={a['ratio']:.1f} -> {a['w']:.0f}x{a['h']:.0f} pixels")

In [ ]:
# === Visualize anchors tiled across an image ===

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
img = create_scene()
ax.imshow(img)

# Place anchors at a grid of feature map locations
# With stride 16, feature map locations map back to every 16th pixel
stride = 50  # Using larger stride for visual clarity
count = 0
for fy in range(stride//2, img.shape[0], stride):
    for fx in range(stride//2, img.shape[1], stride):
        # Draw center dot
        ax.plot(fx, fy, 'r.', markersize=4)
        
        # Draw just one anchor (square, medium) at each location for clarity
        scale = 40
        rect = patches.Rectangle(
            (fx - scale/2, fy - scale/2), scale, scale,
            linewidth=0.8, edgecolor='red', facecolor='none', alpha=0.4
        )
        ax.add_patch(rect)
        count += 1

ax.set_title(f'Anchor Grid: {count} locations shown (only square anchors for clarity)\n'
             f'In reality, 9 different shapes at EACH red dot', fontsize=12, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Dots shown: {count}")
print(f"With 9 anchors each: {count * 9} total anchors")
print(f"In a real 800x800 image with stride 16: {(800//16) * (800//16) * 9:,} anchors!")
print()
print("The model's job: for each of these ~22,500 anchors, predict:")
print("  1. Is there an object here? (objectness score)")
print("  2. If yes, how should I adjust this anchor? (dx, dy, dw, dh)")

## 4. The R-CNN Family Tree -- Evolution of Object Detection

### The Big Picture

The evolution from R-CNN to Faster R-CNN is one of the most elegant stories in deep learning. Each version fixed the biggest bottleneck of the previous one:

| Model | Year | Key Innovation | Speed | Problem Solved |
|-------|------|----------------|-------|----------------|
| **R-CNN** | 2014 | CNN for detection | ~47s/image | First to use deep features for detection |
| **Fast R-CNN** | 2015 | Shared feature computation | ~2s/image | Eliminated redundant CNN passes |
| **Faster R-CNN** | 2015 | Neural region proposals (RPN) | ~0.2s/image | Eliminated slow selective search |

That is a **235x speedup** from R-CNN to Faster R-CNN, while also improving accuracy!

In [ ]:
# === Architecture Evolution Diagram ===

fig, axes = plt.subplots(3, 1, figsize=(16, 18))

# Helper function to draw a box with text
def draw_box(ax, x, y, w, h, text, color, fontsize=9):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15',
                         facecolor=color, edgecolor='#333', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', wrap=True)

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# ============ R-CNN ============
ax = axes[0]
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.set_title('R-CNN (2014) -- "The Pioneer" (~47 seconds per image)', fontsize=14, fontweight='bold')
ax.axis('off')

draw_box(ax, 0.2, 1.2, 1.8, 1.5, 'Input\nImage', '#FFE0B2')
draw_box(ax, 2.8, 1.2, 2.2, 1.5, 'Selective\nSearch\n(~2000 regions)', '#FFCDD2')
draw_box(ax, 5.8, 1.2, 2.2, 1.5, 'CNN\n(run 2000x!!)\nAlexNet/VGG', '#C8E6C9')
draw_box(ax, 8.8, 1.2, 1.8, 1.5, 'SVM\nClassifier', '#B3E5FC')
draw_box(ax, 11.2, 1.2, 2.2, 1.5, 'Bbox\nRegression\n(refine)', '#D1C4E9')

draw_arrow(ax, 2.0, 1.95, 2.8, 1.95)
draw_arrow(ax, 5.0, 1.95, 5.8, 1.95)
draw_arrow(ax, 8.0, 1.95, 8.8, 1.95)
draw_arrow(ax, 10.6, 1.95, 11.2, 1.95)

ax.text(7.0, 0.3, 'BOTTLENECK: CNN runs 2000 times per image (once per region proposal)!',
        fontsize=11, ha='center', color='red', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#FFEBEE', alpha=0.9))

# ============ Fast R-CNN ============
ax = axes[1]
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.set_title('Fast R-CNN (2015) -- "Share the Work" (~2 seconds per image)', fontsize=14, fontweight='bold')
ax.axis('off')

draw_box(ax, 0.2, 1.2, 1.8, 1.5, 'Input\nImage', '#FFE0B2')
draw_box(ax, 2.8, 1.2, 2.2, 1.5, 'CNN\n(run ONCE!)\nShared Features', '#C8E6C9')
draw_box(ax, 2.8, 3.0, 2.2, 0.8, 'Selective\nSearch', '#FFCDD2')
draw_box(ax, 5.8, 1.2, 2.2, 1.5, 'RoI\nPooling\n(crop features)', '#FFF9C4')
draw_box(ax, 8.8, 1.2, 2.0, 1.5, 'FC Layers\nClassifier +\nBox Regressor', '#B3E5FC')
draw_box(ax, 11.5, 1.2, 2.0, 1.5, 'Class +\nBbox', '#D1C4E9')

draw_arrow(ax, 2.0, 1.95, 2.8, 1.95)
draw_arrow(ax, 5.0, 1.95, 5.8, 1.95)
draw_arrow(ax, 8.0, 1.95, 8.8, 1.95)
draw_arrow(ax, 10.8, 1.95, 11.5, 1.95)
draw_arrow(ax, 3.9, 3.0, 6.9, 2.7)  # selective search to RoI pooling

ax.text(7.0, 0.3, 'FIX: CNN runs once; features are shared. BOTTLENECK: Selective Search is still slow & not learnable.',
        fontsize=10, ha='center', color='#E65100', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#FFF3E0', alpha=0.9))

# ============ Faster R-CNN ============
ax = axes[2]
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.set_title('Faster R-CNN (2015) -- "All Neural, All the Time" (~0.2 seconds per image)', fontsize=14, fontweight='bold')
ax.axis('off')

draw_box(ax, 0.2, 1.2, 1.8, 1.5, 'Input\nImage', '#FFE0B2')
draw_box(ax, 2.8, 1.2, 2.2, 1.5, 'Backbone\nCNN\n(ResNet-50)', '#C8E6C9')
draw_box(ax, 5.8, 2.5, 2.2, 1.2, 'RPN\n(learned!)', '#FFCDD2')
draw_box(ax, 5.8, 0.5, 2.2, 1.5, 'RoI\nPooling', '#FFF9C4')
draw_box(ax, 8.8, 1.0, 2.0, 2.0, 'Detection\nHead\n(Cls + Reg)', '#B3E5FC')
draw_box(ax, 11.5, 1.2, 2.0, 1.5, 'Class +\nBbox +\nConf', '#D1C4E9')

draw_arrow(ax, 2.0, 1.95, 2.8, 1.95)
draw_arrow(ax, 5.0, 2.0, 5.8, 1.2)  # to RoI pooling
draw_arrow(ax, 5.0, 2.5, 5.8, 3.0)  # to RPN
draw_arrow(ax, 6.9, 2.5, 6.9, 2.0)  # RPN to RoI pooling
draw_arrow(ax, 8.0, 1.25, 8.8, 1.8)
draw_arrow(ax, 10.8, 2.0, 11.5, 1.95)

ax.text(7.0, 0.0, 'FIX: Region proposals are now a neural network (RPN) -- the ENTIRE pipeline is end-to-end trainable!',
        fontsize=10, ha='center', color='#1B5E20', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#E8F5E9', alpha=0.9))

plt.tight_layout()
plt.show()

print("=== The Evolution Story ===")
print("R-CNN:       'Run CNN on each of 2000 crops.' (slow but first to use deep learning)")
print("Fast R-CNN:  'Run CNN once, crop features.'    (23x faster, but selective search is slow)")
print("Faster R-CNN:'Replace selective search with a neural network (RPN).' (10x faster again!)")
print()
print("Total speedup: ~235x from R-CNN to Faster R-CNN")
print("And Faster R-CNN is MORE accurate, not less!")

## 5. Region Proposal Network (RPN) -- The Key Innovation

### The 12-Year-Old Version

Think of the RPN as a **metal detector on a beach**. Before Faster R-CNN, finding objects was like digging up random spots on the beach hoping to find treasure (Selective Search). The RPN is like sweeping a metal detector over the entire beach first -- it beeps at spots that MIGHT have something interesting, and then you only dig at those spots.

### How It Works (Step by Step)

1. The backbone CNN produces a **feature map** (a compressed, smart version of the image)
2. A small sliding window (3x3 conv) scans across the feature map
3. At each position, it looks at `k` anchor boxes (typically 9)
4. For each anchor, it predicts:
   - **Objectness**: "Is there ANY object here?" (2 scores: object vs. background)
   - **Box deltas**: "How should I adjust this anchor?" (4 values: dx, dy, dw, dh)
5. Keep the top ~300 proposals (after NMS to remove duplicates)

In [ ]:
import torch
import torch.nn as nn

# === Simplified Region Proposal Network ===

class SimpleRPN(nn.Module):
    """
    A simplified Region Proposal Network.
    
    12-year-old version:
    This network slides a 3x3 window across the feature map.
    At each spot, it looks at 9 anchor boxes and says:
      - 'Is there something here?' (objectness score)
      - 'How should I adjust this box?' (box offsets)
    
    Staff-level detail:
    The RPN shares the backbone features with the detection head.
    It adds a 3x3 conv followed by two sibling 1x1 convs:
    one for objectness classification (2k scores) and one for
    bounding box regression (4k coordinates), where k = num anchors.
    """
    
    def __init__(self, in_channels=256, num_anchors=9):
        super().__init__()
        
        # Shared 3x3 conv (the "sliding window")
        self.conv = nn.Conv2d(in_channels, 256, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        
        # Objectness classifier: for each anchor, predict object vs background
        # Output: 2 * num_anchors channels (2 classes per anchor)
        self.cls_layer = nn.Conv2d(256, 2 * num_anchors, kernel_size=1)
        
        # Box regression: for each anchor, predict 4 offsets (dx, dy, dw, dh)
        self.reg_layer = nn.Conv2d(256, 4 * num_anchors, kernel_size=1)
        
        self.num_anchors = num_anchors
    
    def forward(self, feature_map):
        """
        Args:
            feature_map: (batch, channels, H, W) from backbone CNN
        
        Returns:
            objectness: (batch, 2*k, H, W) -- object vs background scores
            box_deltas: (batch, 4*k, H, W) -- bounding box adjustments
        """
        # Shared features
        shared = self.relu(self.conv(feature_map))  # (B, 256, H, W)
        
        # Predict objectness
        objectness = self.cls_layer(shared)  # (B, 2k, H, W)
        
        # Predict box offsets
        box_deltas = self.reg_layer(shared)  # (B, 4k, H, W)
        
        return objectness, box_deltas


# Instantiate and test
rpn = SimpleRPN(in_channels=256, num_anchors=9)

# Simulate a feature map from a backbone (ResNet-50 with stride 16 on 800x800 image)
dummy_feature_map = torch.randn(1, 256, 50, 50)

objectness, box_deltas = rpn(dummy_feature_map)

print("=== Region Proposal Network (RPN) ===")
print(f"\nInput feature map shape:  {dummy_feature_map.shape}")
print(f"  -> batch=1, channels=256, height=50, width=50")
print(f"\nRPN outputs:")
print(f"  Objectness scores: {objectness.shape}")
print(f"    -> 2 scores per anchor x 9 anchors = 18 channels")
print(f"    -> Total predictions: {50*50*9:,} (for all anchor locations)")
print(f"  Box deltas:        {box_deltas.shape}")
print(f"    -> 4 offsets per anchor x 9 anchors = 36 channels")
print()
print("After RPN:")
print("  1. Apply box deltas to anchors to get actual proposals")
print("  2. Sort by objectness score")
print("  3. Apply NMS to remove duplicates")
print("  4. Keep top 300 proposals -> send to detection head")

# Count parameters
total_params = sum(p.numel() for p in rpn.parameters())
print(f"\nRPN parameter count: {total_params:,}")
print("(Tiny compared to the backbone -- the RPN is lightweight!)")

In [ ]:
# === Simulate the Full Faster R-CNN Pipeline Step by Step ===

def simulate_faster_rcnn_pipeline(image_size=(800, 800)):
    """
    Walk through each stage of Faster R-CNN with concrete numbers.
    
    This is the kind of explanation that shows staff-level understanding:
    you can trace exact tensor shapes through the entire pipeline.
    """
    print("=" * 65)
    print("  FASTER R-CNN: COMPLETE PIPELINE WALKTHROUGH")
    print("=" * 65)
    
    H, W = image_size
    stride = 16  # ResNet-50 total stride
    k = 9        # anchors per location
    
    # Stage 0: Input
    print(f"\n--- Stage 0: Input ---")
    print(f"  Image: ({H}, {W}, 3) = {H*W*3:,} pixels")
    
    # Stage 1: Backbone
    feat_h, feat_w = H // stride, W // stride
    feat_channels = 256
    print(f"\n--- Stage 1: Backbone CNN (ResNet-50) ---")
    print(f"  Feature map: ({feat_channels}, {feat_h}, {feat_w})")
    print(f"  Compression: {H*W*3:,} pixels -> {feat_channels*feat_h*feat_w:,} features")
    print(f"  Each feature map cell 'sees' a {stride}x{stride} pixel region")
    
    # Stage 2: RPN
    total_anchors = feat_h * feat_w * k
    print(f"\n--- Stage 2: Region Proposal Network (RPN) ---")
    print(f"  Anchors per location: {k} (3 scales x 3 ratios)")
    print(f"  Total anchors: {feat_h} x {feat_w} x {k} = {total_anchors:,}")
    print(f"  Objectness predictions: {total_anchors:,} x 2 = {total_anchors*2:,}")
    print(f"  Box delta predictions:  {total_anchors:,} x 4 = {total_anchors*4:,}")
    
    # After NMS
    top_k = 300
    print(f"\n  After NMS + top-{top_k} selection:")
    print(f"  {total_anchors:,} anchors -> {top_k} proposals (99% reduction!)")
    
    # Stage 3: RoI Pooling
    roi_size = 7
    print(f"\n--- Stage 3: RoI Pooling ---")
    print(f"  Each proposal (variable size) -> fixed ({roi_size}, {roi_size}) feature grid")
    print(f"  Output: ({top_k}, {feat_channels}, {roi_size}, {roi_size})")
    
    # Stage 4: Detection Head
    num_classes = 3  # background, face, license_plate
    print(f"\n--- Stage 4: Detection Head ---")
    print(f"  Input: {top_k} proposals x {feat_channels*roi_size*roi_size:,} features each")
    print(f"  Classification: {top_k} x {num_classes} class scores")
    print(f"  Box regression: {top_k} x {num_classes} x 4 refined coordinates")
    
    # Final output
    print(f"\n--- Stage 5: Post-Processing ---")
    print(f"  Apply confidence threshold (e.g., 0.5)")
    print(f"  Apply NMS again (class-specific)")
    print(f"  Final output: ~5-20 detections per image (faces + plates)")
    print(f"\n" + "=" * 65)

simulate_faster_rcnn_pipeline()

## 6. One-Stage Detectors: YOLO and SSD

### YOLO -- You Only Look Once

**12-year-old version**: Instead of the two-step process ("find interesting regions" then "classify each one"), YOLO does everything in ONE shot. It divides the image into a grid and says "for each grid cell, what objects are centered here?" It is like taking a test where you answer all questions at once instead of going back to check your answers.

**Staff-level detail**: YOLO divides the image into an SxS grid. Each grid cell predicts B bounding boxes, each with 5 values (x, y, w, h, confidence) plus C class probabilities. The entire prediction is a single tensor of shape (S, S, B*5 + C), produced by a single forward pass through a CNN.

### SSD -- Single Shot MultiBox Detector

**12-year-old version**: SSD is like YOLO but smarter about sizes. It looks at the image at multiple zoom levels -- like looking through binoculars, then regular glasses, then a magnifying glass. Small objects are caught by the magnifying glass (early layers), big objects by the binoculars (later layers).

**Staff-level detail**: SSD uses multi-scale feature maps (extracted from different layers of the backbone) and applies anchor-based predictions at each scale. This naturally handles objects of different sizes.

In [ ]:
# === Visualize YOLO's Grid-Based Detection ===

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
img = create_scene()

# --- Panel 1: YOLO Grid ---
ax = axes[0]
ax.imshow(img)
S = 7  # 7x7 grid (like YOLOv1)
cell_h = img.shape[0] / S
cell_w = img.shape[1] / S

# Draw grid
for i in range(1, S):
    ax.axhline(i * cell_h, color='white', linewidth=0.8, alpha=0.6)
    ax.axvline(i * cell_w, color='white', linewidth=0.8, alpha=0.6)

# Highlight cells responsible for objects
# Person center is roughly at (85, 90) -> grid cell (1, 2)
person_cell = patches.Rectangle((1*cell_w, 2*cell_h), cell_w, cell_h,
                                 linewidth=3, edgecolor='red', facecolor='red', alpha=0.3)
ax.add_patch(person_cell)
ax.text(1*cell_w + cell_w/2, 2*cell_h + cell_h/2, 'person', color='red',
        ha='center', va='center', fontsize=8, fontweight='bold')

# Car center at (~280, 120) -> grid cell (4, 3)
car_cell = patches.Rectangle((4*cell_w, 2*cell_h), cell_w, cell_h,
                              linewidth=3, edgecolor='blue', facecolor='blue', alpha=0.3)
ax.add_patch(car_cell)
ax.text(4*cell_w + cell_w/2, 2*cell_h + cell_h/2, 'car', color='blue',
        ha='center', va='center', fontsize=8, fontweight='bold')

ax.set_title(f'YOLO: {S}x{S} Grid\nEach cell predicts objects centered in it', fontsize=12, fontweight='bold')
ax.axis('off')

# --- Panel 2: SSD Multi-Scale ---
ax = axes[1]
scale_sizes = [(38, 38), (19, 19), (10, 10), (5, 5), (3, 3), (1, 1)]
scale_colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db', '#9b59b6']

for i, ((h, w), color) in enumerate(zip(scale_sizes, scale_colors)):
    bar_height = 0.6
    y_pos = 5 - i
    bar_width = w / 38 * 8  # Scale for visualization
    rect = patches.Rectangle((1, y_pos), bar_width, bar_height,
                              facecolor=color, edgecolor='#333', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(bar_width + 1.3, y_pos + bar_height/2, f'{h}x{w}  ({h*w*6} anchors)',
            va='center', fontsize=10)

ax.set_xlim(0, 12)
ax.set_ylim(0, 6.5)
ax.set_title('SSD: Multi-Scale Feature Maps\nDifferent layers detect different sizes', fontsize=12, fontweight='bold')
ax.text(6, 6.2, 'Small objects (early layers) -> Large objects (later layers)',
        ha='center', fontsize=9, style='italic')
ax.axis('off')

# --- Panel 3: Speed vs Accuracy ---
ax = axes[2]
models = {
    'R-CNN': (0.02, 30, '#e74c3c'),
    'Fast R-CNN': (0.5, 35, '#e67e22'),
    'Faster R-CNN': (5, 42, '#2ecc71'),
    'SSD': (22, 38, '#3498db'),
    'YOLOv3': (35, 37, '#9b59b6'),
    'YOLOv5': (60, 40, '#e91e63'),
}

for name, (fps, map_score, color) in models.items():
    ax.scatter(fps, map_score, s=200, c=color, edgecolors='#333', linewidths=1.5, zorder=3)
    offset_x = 1 if fps < 50 else -5
    offset_y = 0.8
    ax.annotate(name, (fps, map_score), textcoords='offset points',
                xytext=(offset_x, offset_y+5), fontsize=9, fontweight='bold')

ax.set_xlabel('Speed (FPS)', fontsize=12)
ax.set_ylabel('mAP on COCO', fontsize=12)
ax.set_title('Speed vs Accuracy Trade-off', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_xscale('log')

# Add arrow annotation for our choice
ax.annotate('Our choice\n(offline, accuracy first)',
            xy=(5, 42), xytext=(15, 44),
            arrowprops=dict(arrowstyle='->', color='green', lw=2),
            fontsize=10, color='green', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# === When to Use Which Architecture: Decision Framework ===

decision_matrix = {
    "Scenario": [
        "Google Street View (this system)",
        "Self-driving car",
        "Security camera",
        "Photo app face filter",
        "Retail shelf analysis",
    ],
    "Latency Requirement": [
        "None (offline batch)",
        "Real-time (<50ms)",
        "Near real-time (<200ms)",
        "Real-time (<30ms)",
        "Moderate (<1s)",
    ],
    "Accuracy Priority": [
        "MAX (privacy)",
        "Very High (safety)",
        "High",
        "Moderate",
        "High",
    ],
    "Best Architecture": [
        "Faster R-CNN",
        "YOLO (latest version)",
        "SSD or YOLOv5",
        "YOLO (mobile variant)",
        "Faster R-CNN or SSD",
    ],
    "Why": [
        "No speed constraint; max recall matters",
        "Must be real-time; lives at stake",
        "Balance speed and accuracy",
        "Must run on phone GPU",
        "Accuracy matters; batch processing OK",
    ],
}

print("=== Architecture Decision Framework ===")
print(f"{'Scenario':<32} {'Latency':<22} {'Accuracy':<14} {'Architecture':<18} {'Why'}")
print("-" * 120)
for i in range(len(decision_matrix["Scenario"])):
    print(f"{decision_matrix['Scenario'][i]:<32} "
          f"{decision_matrix['Latency Requirement'][i]:<22} "
          f"{decision_matrix['Accuracy Priority'][i]:<14} "
          f"{decision_matrix['Best Architecture'][i]:<18} "
          f"{decision_matrix['Why'][i]}")

print("\n" + "=" * 70)
print("RULE OF THUMB:")
print("  Need speed?     -> One-stage (YOLO, SSD)")
print("  Need accuracy?  -> Two-stage (Faster R-CNN)")
print("  Need both?      -> Latest YOLO version (v5/v7/v8) or EfficientDet")
print("  Want simplicity?-> DETR (no anchors, no NMS, end-to-end)")

## 7. The Combined Loss Function

### The 12-Year-Old Version

The model has two jobs at the same time:
1. **Job 1 (Classification)**: "Is this a face, a license plate, or nothing?" -- like a multiple-choice quiz
2. **Job 2 (Localization)**: "Where exactly is it? How big is it?" -- like measuring with a ruler

Each job has its own grade (loss). We combine them into one final grade:

```
Final Loss = Classification Loss + lambda * Localization Loss
```

Where `lambda` is a knob that controls which job the model tries harder on.

### Staff-Level Detail

- **Classification loss**: Cross-entropy for multi-class (face / plate / background)
- **Localization loss**: Smooth L1 (Huber) loss for bounding box regression -- more robust than MSE to outliers
- **Lambda**: Typically set to 1.0 (equal weighting), but can be tuned

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# === Object Detection Combined Loss Function ===

class DetectionLoss(nn.Module):
    """
    Combined loss for object detection.
    
    L = L_cls + lambda * L_reg
    
    12-year-old version:
    Two grades combined into one:
      - Classification grade: Did you correctly name the object?
      - Localization grade: Did you draw the box in the right place?
    
    Staff-level detail:
    - Classification: Cross-entropy loss over K classes
    - Localization: Smooth L1 (Huber) loss over (dx, dy, dw, dh)
    - Smooth L1 is preferred over MSE because it is less sensitive
      to outliers (very wrong predictions do not blow up the gradient)
    """
    
    def __init__(self, lambda_reg=1.0):
        super().__init__()
        self.lambda_reg = lambda_reg
        self.cls_loss_fn = nn.CrossEntropyLoss()
        self.reg_loss_fn = nn.SmoothL1Loss()
    
    def forward(self, cls_preds, cls_targets, box_preds, box_targets, pos_mask):
        """
        Args:
            cls_preds: (N, num_classes) predicted class logits
            cls_targets: (N,) ground truth class indices
            box_preds: (N, 4) predicted box deltas
            box_targets: (N, 4) ground truth box deltas
            pos_mask: (N,) boolean mask for positive (non-background) samples
                      We only compute regression loss on positive samples!
        """
        # Classification loss (on ALL predictions)
        L_cls = self.cls_loss_fn(cls_preds, cls_targets)
        
        # Regression loss (only on POSITIVE predictions -- no point regressing background)
        if pos_mask.sum() > 0:
            L_reg = self.reg_loss_fn(box_preds[pos_mask], box_targets[pos_mask])
        else:
            L_reg = torch.tensor(0.0)
        
        # Combined loss
        total_loss = L_cls + self.lambda_reg * L_reg
        
        return total_loss, L_cls, L_reg


# === Demonstrate with simulated predictions ===
torch.manual_seed(42)

num_predictions = 100  # 100 anchor predictions
num_classes = 3        # background, face, plate

# Simulated predictions
cls_preds = torch.randn(num_predictions, num_classes)
cls_targets = torch.zeros(num_predictions, dtype=torch.long)  # Mostly background
cls_targets[:10] = 1   # 10 face predictions
cls_targets[10:15] = 2 # 5 plate predictions

box_preds = torch.randn(num_predictions, 4) * 0.1
box_targets = torch.randn(num_predictions, 4) * 0.1

pos_mask = cls_targets > 0  # Positive = non-background

# Compute loss
loss_fn = DetectionLoss(lambda_reg=1.0)
total, cls, reg = loss_fn(cls_preds, cls_targets, box_preds, box_targets, pos_mask)

print("=== Detection Loss Computation ===")
print(f"\n  Total predictions:    {num_predictions}")
print(f"  Positive (object):    {pos_mask.sum().item()} ({pos_mask.float().mean():.0%})")
print(f"  Negative (background):{(~pos_mask).sum().item()} ({(~pos_mask).float().mean():.0%})")
print(f"\n  Classification Loss (L_cls): {cls.item():.4f}")
print(f"  Regression Loss (L_reg):     {reg.item():.4f}")
print(f"  Lambda:                      {loss_fn.lambda_reg}")
print(f"  Total Loss = {cls.item():.4f} + {loss_fn.lambda_reg} * {reg.item():.4f} = {total.item():.4f}")
print()
print("KEY INSIGHT: Regression loss is ONLY computed on positive samples.")
print("Why? It makes no sense to regress a bounding box for 'background'.")
print("This is a very common interview question!")

In [ ]:
# === Visualize Smooth L1 vs MSE Loss ===

x = np.linspace(-3, 3, 300)

# MSE loss
mse = x ** 2

# Smooth L1 (Huber) loss
smooth_l1 = np.where(np.abs(x) < 1, 0.5 * x**2, np.abs(x) - 0.5)

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(x, mse, 'r-', linewidth=2.5, label='MSE (L2 Loss)')
ax.plot(x, smooth_l1, 'b-', linewidth=2.5, label='Smooth L1 (Huber Loss)')

ax.set_xlabel('Prediction Error', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Why Smooth L1 Is Better Than MSE for Bounding Box Regression', fontsize=13, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# Annotate the key difference
ax.annotate('MSE explodes\nfor large errors!',
            xy=(2.5, 6.25), xytext=(1.5, 7.5),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=11, color='red', fontweight='bold')

ax.annotate('Smooth L1 stays\nlinear (controlled)',
            xy=(2.5, 2.0), xytext=(0.5, 3.5),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2),
            fontsize=11, color='blue', fontweight='bold')

plt.tight_layout()
plt.show()

print("Why Smooth L1 beats MSE for bounding boxes:")
print("  - MSE squares the error: a box that is 10px off gets 100 loss (explosive!)")
print("  - Smooth L1: for small errors (<1), behaves like MSE (quadratic)")
print("                for large errors (>1), behaves like L1 (linear -- controlled)")
print("  - This prevents large gradients from a few bad predictions from")
print("    destabilizing training.")

## 8. Data Augmentation for Object Detection

### The 12-Year-Old Version

Data augmentation for detection is TRICKIER than for classification. When you flip or rotate a photo, you also need to flip or rotate all the bounding boxes! If you forget, the model will learn that a face is on the left side of the image when the face is actually on the right (because you flipped the image but not the box).

Think of it like this: if you flip a photo of yourself in a mirror, your face moves from the left side to the right side of the photo. The bounding box around your face needs to move too!

### Staff-Level Detail

All geometric transforms (flip, rotate, crop, affine) must be applied jointly to both the image AND the bounding box annotations. Photometric transforms (brightness, contrast, noise) do NOT change box coordinates.

In [ ]:
# === Data Augmentation with Bounding Box Transforms ===

def create_face_image():
    """Create a synthetic image with a face-like region."""
    img = np.zeros((200, 300, 3), dtype=np.uint8)
    for i in range(200):
        img[i, :] = [100 + i//4, 150 - i//5, 180 - i//4]
    # Face region
    img[50:130, 80:150] = [220, 180, 150]
    img[70:80, 95:105] = [50, 50, 50]   # left eye
    img[70:80, 125:135] = [50, 50, 50]  # right eye
    img[100:108, 105:130] = [180, 100, 100]  # mouth
    # License plate
    img[150:170, 180:260] = [240, 240, 240]
    img[155:165, 190:250] = [40, 40, 40]
    
    bboxes = [
        {'bbox': [80, 50, 70, 80], 'class': 'face'},     # x, y, w, h
        {'bbox': [180, 150, 80, 20], 'class': 'plate'},
    ]
    return img, bboxes


def augment_horizontal_flip(img, bboxes):
    """Flip image and ALL bounding boxes."""
    flipped_img = np.fliplr(img).copy()
    W = img.shape[1]
    flipped_bboxes = []
    for b in bboxes:
        x, y, w, h = b['bbox']
        new_x = W - x - w  # Mirror x coordinate
        flipped_bboxes.append({'bbox': [new_x, y, w, h], 'class': b['class']})
    return flipped_img, flipped_bboxes


def augment_random_crop(img, bboxes, crop_frac=0.8):
    """Crop a portion of the image, adjusting bboxes."""
    H, W = img.shape[:2]
    new_H, new_W = int(H * crop_frac), int(W * crop_frac)
    
    # Random crop position
    np.random.seed(42)
    y_start = np.random.randint(0, H - new_H)
    x_start = np.random.randint(0, W - new_W)
    
    cropped_img = img[y_start:y_start+new_H, x_start:x_start+new_W].copy()
    
    cropped_bboxes = []
    for b in bboxes:
        x, y, w, h = b['bbox']
        # Shift bbox to crop coordinates
        new_x = x - x_start
        new_y = y - y_start
        # Clip to crop boundaries
        new_x = max(0, new_x)
        new_y = max(0, new_y)
        new_w = min(new_x + w, new_W) - new_x
        new_h = min(new_y + h, new_H) - new_y
        
        # Only keep bbox if enough of it is visible
        original_area = w * h
        new_area = new_w * new_h
        if new_area > 0.3 * original_area and new_w > 5 and new_h > 5:
            cropped_bboxes.append({'bbox': [new_x, new_y, new_w, new_h], 'class': b['class']})
    
    return cropped_img, cropped_bboxes


def augment_brightness(img, bboxes, factor=1.4):
    """Change brightness. Bboxes stay the same (photometric transform)."""
    bright_img = np.clip(img.astype(float) * factor, 0, 255).astype(np.uint8)
    return bright_img, bboxes  # Bboxes unchanged!


# Generate all augmentations
img, bboxes = create_face_image()
flip_img, flip_bboxes = augment_horizontal_flip(img, bboxes)
crop_img, crop_bboxes = augment_random_crop(img, bboxes)
bright_img, bright_bboxes = augment_brightness(img, bboxes)

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
augmentations = [
    ('Original', img, bboxes),
    ('Horizontal Flip', flip_img, flip_bboxes),
    ('Random Crop', crop_img, crop_bboxes),
    ('Brightness +40%', bright_img, bright_bboxes),
]

for ax, (title, im, bbs) in zip(axes, augmentations):
    ax.imshow(im)
    for b in bbs:
        x, y, w, h = b['bbox']
        color = 'red' if b['class'] == 'face' else 'blue'
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x, y - 3, b['class'], color=color, fontsize=9, fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('Data Augmentation for Object Detection\n(Notice: bounding boxes transform WITH the image!)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("=== Augmentation Types ===")
print()
print("GEOMETRIC (change position -> MUST transform bboxes):")
print("  - Horizontal flip: mirror x coordinates")
print("  - Random crop: shift + clip bboxes, drop if <30% visible")
print("  - Rotation: apply rotation matrix to bbox corners")
print("  - Affine transform: apply same affine to bboxes")
print()
print("PHOTOMETRIC (change appearance -> bboxes UNCHANGED):")
print("  - Brightness, contrast, saturation")
print("  - Random noise")
print("  - Color jitter")
print()
print("COMMON MISTAKE: Forgetting to transform bboxes after geometric augmentation!")
print("This is a subtle bug that silently ruins model accuracy.")

## Interview Cheat Sheet: Object Detection Deep Dive

### 30-Second Summary

"Object detection combines localization and classification. Two-stage detectors like Faster R-CNN use a Region Proposal Network to suggest candidate regions, then classify each one. One-stage detectors like YOLO do both in a single pass. For an offline privacy system like Street View blurring, we choose Faster R-CNN because accuracy and recall matter more than speed. The model uses anchor boxes at each feature map location and predicts objectness + box offsets. The loss combines cross-entropy for classification and Smooth L1 for bounding box regression."

### Key Phrases to Drop

- "Anchor boxes provide translation-invariant priors at each spatial location"
- "The RPN shares convolutional features with the detection head -- this is what makes Faster R-CNN efficient"
- "Smooth L1 loss is robust to outliers, preventing gradient explosion from large box errors"
- "Data augmentation for detection requires joint transformation of images AND bounding boxes"
- "The combined loss L = L_cls + lambda * L_reg balances classification and localization objectives"

### Common Follow-Up Questions

| Question | Key Answer Points |
|----------|-------------------|
| Why not YOLO for Street View? | Offline processing = no latency constraint, so we pick max accuracy |
| What is the RPN? | A small CNN that slides over the feature map, predicting objectness + box offsets for each anchor |
| Why Smooth L1 over MSE? | MSE squares errors, so a 10px error becomes 100 loss -- Smooth L1 stays linear for large errors |
| How many anchors? | Typically 9 per location (3 scales x 3 ratios), so ~22,500 for a 50x50 feature map |
| Why not compute regression loss on background? | No point regressing a box for "nothing" -- waste of computation and confuses the model |

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)